<a href="https://colab.research.google.com/github/JorgeAccardi/auscultacion-presa/blob/main/Preprocesamiento_EDA_Plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import io
import base64
from datetime import datetime
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# --- Configuración y estilos ---
instrumentos = [
    "puntos_fijos_mi",
    "puntos_fijos_md",
    "inclinometros",
    "asentamiento",
    "piezometros_electricos",
    "piezometros_casagrande",
    "freatimetros",
    "extensometro"
]
datos_csv = {inst: pd.DataFrame() for inst in instrumentos}
datos_xlsx = {inst: pd.DataFrame() for inst in instrumentos}

def detectar_instrumento(nombre):
    nombre = nombre.lower()
    if "puntosfijos" in nombre or "pf" in nombre:
        if "mi" in nombre:
            return "puntos_fijos_mi"
        elif "md" in nombre:
            return "puntos_fijos_md"
        else:
            return "puntos_fijos_mi"
    elif "incli" in nombre:
        return "inclinometros"
    elif "as" in nombre:
        return "asentamiento"
    elif "pe" in nombre:
        return "piezometros_electricos"
    elif "pcg" in nombre:
        return "piezometros_casagrande"
    elif "frea" in nombre:
        return "freatimetros"
    elif "ext" in nombre:
        return "extensometro"
    return None

# Widgets globales para mantener la UI estable
upload_widget = widgets.FileUpload(
    accept='.csv,.xlsx',
    multiple=True,
    description='Subir archivos',
    style={'button_color': 'lightblue'},
    layout=widgets.Layout(width="350px")
)
output_carga = widgets.Output()
output_tabla = widgets.Output()

instrumento_selector = widgets.Dropdown(
    options=instrumentos,
    description='Instrumento:',
    layout=widgets.Layout(width='260px')
)
origen_selector = widgets.Dropdown(
    options=['csv', 'xlsx'],
    description='Origen:',
    layout=widgets.Layout(width='160px')
)
boton_ver = widgets.Button(
    description='👁️ Ver',
    button_style='success',
    icon='eye',
    layout=widgets.Layout(width='100px')
)
boton_descargar = widgets.Button(
    description='💾 Descargar',
    button_style='info',
    icon='download',
    layout=widgets.Layout(width='130px')
)

# --- Función de carga y progreso ---
def cargar_archivos(change):
    with output_carga:
        clear_output(wait=True)
        archivos = upload_widget.value
        if not archivos:
            display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No se subió ningún archivo.</div>"))
            return

        barra_progreso = widgets.FloatProgress(
            value=0, min=0, max=100, description='Progreso:', bar_style='info',
            layout=widgets.Layout(width='80%')
        )
        etiqueta_progreso = widgets.Label(value="0% completado")
        display(barra_progreso, etiqueta_progreso)

        total = len(archivos)
        archivos_exitosos = 0

        for i, (nombre_archivo, archivo_info) in enumerate(archivos.items(), start=1):
            try:
                extension = nombre_archivo.split('.')[-1].lower()
                instrumento = detectar_instrumento(nombre_archivo)
                contenido = archivo_info['content']

                if not instrumento:
                    display(HTML(f"<span style='color:#b71c1c;'>❌ Instrumento no reconocido en archivo: {nombre_archivo}</span>"))
                    continue

                if extension == 'csv':
                    try:
                        try:
                            df = pd.read_csv(io.BytesIO(contenido), encoding='utf-8')
                        except UnicodeDecodeError:
                            df = pd.read_csv(io.BytesIO(contenido), encoding='latin-1')
                        if not df.empty:
                            datos_csv[instrumento] = pd.concat([datos_csv[instrumento], df], ignore_index=True)
                            archivos_exitosos += 1
                            display(HTML(f"<span style='color:#388e3c;'>✔️ {nombre_archivo} cargado como CSV ({len(df)} filas)</span>"))
                        else:
                            display(HTML(f"<span style='color:#ffa000;'>⚠️ CSV vacío: {nombre_archivo}</span>"))
                    except Exception as e:
                        display(HTML(f"<span style='color:#b71c1c;'>❌ Error leyendo CSV {nombre_archivo}: {str(e)}</span>"))
                elif extension == 'xlsx':
                    try:
                        df = pd.read_excel(io.BytesIO(contenido))
                        if not df.empty:
                            datos_xlsx[instrumento] = pd.concat([datos_xlsx[instrumento], df], ignore_index=True)
                            archivos_exitosos += 1
                            display(HTML(f"<span style='color:#388e3c;'>✔️ {nombre_archivo} cargado como XLSX ({len(df)} filas)</span>"))
                        else:
                            display(HTML(f"<span style='color:#ffa000;'>⚠️ XLSX vacío: {nombre_archivo}</span>"))
                    except Exception as e:
                        display(HTML(f"<span style='color:#b71c1c;'>❌ Error leyendo XLSX {nombre_archivo}: {str(e)}</span>"))
                else:
                    display(HTML(f"<span style='color:#b71c1c;'>⚠️ Extensión no soportada: {nombre_archivo}</span>"))
            except Exception as e:
                display(HTML(f"<span style='color:#b71c1c;'>❌ Error general en {nombre_archivo}: {str(e)}</span>"))
            porcentaje = (i / total) * 100
            barra_progreso.value = porcentaje
            etiqueta_progreso.value = f"{porcentaje:.0f}% completado"

        barra_progreso.bar_style = 'success'
        etiqueta_progreso.value = f"✅ {archivos_exitosos}/{total} archivos procesados exitosamente"
        resumen = "<ul>"
        for inst in instrumentos:
            csv_count = len(datos_csv[inst])
            xlsx_count = len(datos_xlsx[inst])
            if csv_count > 0 or xlsx_count > 0:
                resumen += f"<li><b>{inst}</b>: {csv_count} filas CSV, {xlsx_count} filas XLSX</li>"
        resumen += "</ul>"
        display(HTML(f"<div style='margin-top:10px;'><b>Resumen de datos:</b>{resumen}</div>"))

# --- Función para mostrar tabla o mensaje ---
def ver_datos(b):
    with output_tabla:
        clear_output(wait=True)
        instrumento = instrumento_selector.value
        origen = origen_selector.value
        df = datos_csv[instrumento] if origen == 'csv' else datos_xlsx[instrumento]
        if df.empty:
            display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No hay datos disponibles para el instrumento y origen seleccionados.</div>"))
        else:
            display(HTML(f"<div style='margin-bottom:10px;'><b>{instrumento.replace('_',' ').title()} ({origen.upper()})</b> - <span style='color:#388e3c'>Filas: {len(df)} | Columnas: {len(df.columns)}</span></div>"))
            display(df.head(5))  # AJUSTE: solo 5 registros

# --- Función para descargar con enlace bonito ---
def descargar_datos(b):
    with output_tabla:
        clear_output(wait=True)
        instrumento = instrumento_selector.value
        origen = origen_selector.value
        df = datos_csv[instrumento] if origen == 'csv' else datos_xlsx[instrumento]
        if df.empty:
            display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No hay datos para descargar.</div>"))
            return
        fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
        extension = 'csv' if origen == 'csv' else 'xlsx'
        nombre_archivo = f"{instrumento}_{origen}_{fecha_actual}.{extension}"
        buffer = io.BytesIO()
        try:
            if extension == 'csv':
                df.to_csv(buffer, index=False, encoding='utf-8')
                mime = "text/csv"
            else:
                # openpyxl es necesario para xlsx, pero en Colab/Jupyter viene instalado
                df.to_excel(buffer, index=False, engine='openpyxl')
                mime = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
            buffer.seek(0)
            b64 = base64.b64encode(buffer.read()).decode()
            # NOTA: El enlace debe estar en una sola línea sin saltos para funcionar bien
            html = f"""
            <div style='padding:12px 18px;background:#e3f2fd;border:1.5px solid #2196f3;border-radius:7px;max-width:420px;'>
                <b>Descarga lista:</b><br>
                <span style='color:#1565c0'><b>{nombre_archivo}</b></span><br>
                <a download="{nombre_archivo}" href="data:{mime};base64,{b64}" target="_blank" style="display:inline-block;margin-top:10px;padding:10px 18px;background:#2196f3;color:white;text-decoration:none;border-radius:4px;font-weight:bold;">
                   📥 Descargar archivo
                </a>
            </div>
            """
            display(HTML(html))
            display(HTML("<span style='color:#388e3c;'>✔️ Haz clic en el botón para guardar el archivo en tu PC.</span>"))
        except Exception as e:
            display(HTML(f"<span style='color:#b71c1c;'>❌ Error al preparar la descarga: {str(e)}</span>"))

# --- Conectar eventos (solo una vez) ---
upload_widget.observe(cargar_archivos, names='value')
boton_ver.on_click(ver_datos)
boton_descargar.on_click(descargar_datos)

# --- Mostrar interfaz visual limpia y separada ---
display(HTML("""
<div style='margin-bottom:15px;'>
    <h2 style='color:#1976d2;margin:0 0 4px 0;'>📈 Sistema de gestión de datos de instrumentos</h2>
    <span style='color:#555;'>Carga, visualización y descarga de archivos CSV/XLSX</span>
</div>
"""))
display(HTML("<b>1. Subí tus archivos CSV/XLSX:</b>"))
display(upload_widget)
display(output_carga)
display(HTML("<hr style='margin:20px 0 10px 0;'>"))
display(HTML("<b>2. Seleccioná instrumento y origen de datos:</b>"))
display(widgets.HBox([instrumento_selector, origen_selector, boton_ver, boton_descargar]))
display(output_tabla)

FileUpload(value={}, accept='.csv,.xlsx', description='Subir archivos', layout=Layout(width='350px'), multiple…

Output()

Output()

In [10]:
# 💡 Esto instalará versiones compatibles
!pip install -U plotly==6.1.1 kaleido==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.0 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Suprimir warnings para visualización limpia
warnings.filterwarnings('ignore')

# =============================================================================
# MÓDULO DE PREPROCESAMIENTO Y VISUALIZACIÓN
# =============================================================================

class PreprocesamientoDatos:
    def __init__(self, datos_csv, datos_xlsx):
        self.datos_csv = datos_csv
        self.datos_xlsx = datos_xlsx
        self.instrumentos = [
            "Puntos Fijos", "Piezómetros Eléctricos", "Piezómetros Casagrande",
            "Inclinómetros", "Celdas de Asentamiento", "Freatímetros", "Extensómetros"
        ]

    def obtener_datos_filtrados(self, instrumento, origen, **kwargs):
        """Obtiene los datos filtrados según los criterios del instrumento"""
        try:
            if instrumento == "Puntos Fijos":
                df_mi = self.datos_csv["puntos_fijos_mi"] if origen == "CSV" else self.datos_xlsx["puntos_fijos_mi"]
                df_md = self.datos_csv["puntos_fijos_md"] if origen == "CSV" else self.datos_xlsx["puntos_fijos_md"]
                datasets = {"Margen Izquierda (MI)": df_mi.copy(), "Margen Derecha (MD)": df_md.copy()}
                margen = kwargs.get('margen', "Todos")
                if margen == "Todos":
                    df = pd.concat([df for df in datasets.values() if not df.empty], ignore_index=True)
                else:
                    df = datasets[margen]
                if kwargs.get('punto') and kwargs['punto'] != "Todos":
                    df = df[df['INSTRUMENTO'] == kwargs['punto']].copy()
                if kwargs.get('anio') and kwargs['anio'] != "Todos":
                    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                    df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

            elif instrumento == "Piezómetros Eléctricos":
                df = (self.datos_csv["piezometros_electricos"] if origen == "CSV" else self.datos_xlsx["piezometros_electricos"]).copy()
                if kwargs.get('progresiva') and kwargs['progresiva'] != "Todos":
                    df = df[df['PROGRESIVA'] == kwargs['progresiva']].copy()
                if kwargs.get('piezometro') and kwargs['piezometro'] != "Todos":
                    df = df[df['PIEZOMETRO'] == kwargs['piezometro']].copy()
                if kwargs.get('anio') and kwargs['anio'] != "Todos":
                    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                    df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

            elif instrumento == "Piezómetros Casagrande":
                df = (self.datos_csv["piezometros_casagrande"] if origen == "CSV" else self.datos_xlsx["piezometros_casagrande"]).copy()
                if kwargs.get('margen') and kwargs['margen'] != "Todos":
                    df = df[df['MARGEN'] == kwargs['margen']].copy()
                if kwargs.get('piezometro') and kwargs['piezometro'] != "Todos":
                    df = df[df['PIEZOMETRO'] == kwargs['piezometro']].copy()
                if kwargs.get('anio') and kwargs['anio'] != "Todos":
                    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                    df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

            elif instrumento == "Inclinómetros":
                df = (self.datos_csv["inclinometros"] if origen == "CSV" else self.datos_xlsx["inclinometros"]).copy()
                if kwargs.get('inclinometro') and kwargs['inclinometro'] != "Todos":
                    df = df[df['Inclinometro'] == kwargs['inclinometro']].copy()
                if kwargs.get('anio') and kwargs['anio'] != "Todos":
                    df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
                    df = df[df['Fecha'].dt.year == int(kwargs['anio'])].copy()
                if kwargs.get('eje') and kwargs['eje'] != "Todos":
                    df = df[['Fecha', 'Profundidad', kwargs['eje']]].copy()
                else:
                    df = df[['Fecha', 'Profundidad', 'A+', 'A-', 'B+', 'B-']].copy()

            elif instrumento == "Celdas de Asentamiento":
                df = (self.datos_csv["asentamiento"] if origen == "CSV" else self.datos_xlsx["asentamiento"]).copy()
                if kwargs.get('progresiva') and kwargs['progresiva'] != "Todos":
                    df = df[df['PROGRESIVA'] == kwargs['progresiva']].copy()
                if kwargs.get('celda') and kwargs['celda'] != "Todos":
                    df = df[df['CELDA_DE_ASENTAMIENTO'] == kwargs['celda']].copy()
                if kwargs.get('anio') and kwargs['anio'] != "Todos":
                    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                    df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

            elif instrumento == "Freatímetros":
                df = (self.datos_csv["freatimetros"] if origen == "CSV" else self.datos_xlsx["freatimetros"]).copy()
                if kwargs.get('freatimetro') and kwargs['freatimetro'] != "Todos":
                    df = df[df['FREATIMETRO'] == kwargs['freatimetro']].copy()
                if kwargs.get('anio') and kwargs['anio'] != "Todos":
                    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                    df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

            elif instrumento == "Extensómetros":
                df = (self.datos_csv["extensometro"] if origen == "CSV" else self.datos_xlsx["extensometro"]).copy()
                if kwargs.get('extensometro') and kwargs['extensometro'] != "Todos":
                    df = df[df['EXTENSOMETRO'] == kwargs['extensometro']].copy()
                if kwargs.get('anio') and kwargs['anio'] != "Todos":
                    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                    df = df[df['FECHA'].dt.year == int(kwargs['anio'])].copy()

            return df.copy()
        except Exception as e:
            print(f"Error al obtener datos filtrados: {str(e)}")
            return pd.DataFrame()

    def detectar_columnas_temporales(self, df):
        """Detecta automáticamente columnas que pueden ser fechas"""
        columnas_temporales = []
        for col in df.columns:
            if df[col].dtype == 'object':
                try:
                    muestra = df[col].dropna().iloc[:5]
                    pd.to_datetime(muestra)
                    columnas_temporales.append(col)
                except:
                    continue
        return columnas_temporales

    def detectar_columnas_numericas(self, df):
        """Detecta columnas numéricas excluyendo las temporales"""
        numericas = df.select_dtypes(include=[np.number]).columns.tolist()
        return numericas

    def imputar_valores_faltantes(self, df, metodo='linear'):
        """Imputa valores faltantes en columnas numéricas"""
        numericas = self.detectar_columnas_numericas(df)
        if not numericas:
            return df

        if metodo == 'linear':
            df[numericas] = df[numericas].interpolate(method='linear', limit_direction='both')
        elif metodo == 'mean':
            df[numericas] = df[numericas].fillna(df[numericas].mean())
        elif metodo == 'median':
            df[numericas] = df[numericas].fillna(df[numericas].median())

        # Rellenar valores restantes con media
        df[numericas] = df[numericas].fillna(df[numericas].mean())
        return df

    def tratar_outliers(self, df, metodo='clip', umbral_iqr=1.5):
        """Trata outliers en columnas numéricas"""
        numericas = self.detectar_columnas_numericas(df)
        if not numericas:
            return df

        for col in numericas:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - umbral_iqr * IQR
            upper_bound = Q3 + umbral_iqr * IQR

            if metodo == 'clip':
                df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
            elif metodo == 'remove':
                df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

        return df

    def normalizar_datos(self, df, metodo='standard'):
        """Normaliza o estandariza columnas numéricas"""
        numericas = self.detectar_columnas_numericas(df)
        if not numericas:
            return df

        if metodo == 'standard':
            scaler = StandardScaler()
        elif metodo == 'minmax':
            scaler = MinMaxScaler()
        else:
            return df

        df[numericas] = scaler.fit_transform(df[numericas])
        return df

    def ingenieria_caracteristicas(self, df, ventana=7):
        """Genera características derivadas como tasas de cambio y promedios móviles"""
        numericas = self.detectar_columnas_numericas(df)
        if not numericas:
            return df

        for col in numericas:
            df[f'{col}_diff'] = df[col].diff()
            df[f'{col}_rolling_mean_{ventana}'] = df[col].rolling(window=ventana, min_periods=1).mean()
            df[f'{col}_rolling_std_{ventana}'] = df[col].rolling(window=ventana, min_periods=1).std()

        return df

    def sincronizar_temporal(self, dfs, freq='D'):
        """Sincroniza temporalmente múltiples DataFrames"""
        if not dfs:
            return []

        fecha_min = min([df['FECHA'].min() for df in dfs if 'FECHA' in df and not df.empty])
        fecha_max = max([df['FECHA'].max() for df in dfs if 'FECHA' in df and not df.empty])
        index_temporal = pd.date_range(fecha_min, fecha_max, freq=freq)

        dfs_sincronizados = []
        for df in dfs:
            if df.empty or 'FECHA' not in df:
                continue
            df = df.set_index('FECHA').reindex(index_temporal).interpolate(method='linear').reset_index()
            df = df.rename(columns={'index': 'FECHA'})
            dfs_sincronizados.append(df)

        return dfs_sincronizados

    def preprocesar_datos(self, instrumento, origen, metodo_imputacion='linear',
                         metodo_outliers='clip', metodo_normalizacion='standard',
                         ventana_caracteristicas=7, **kwargs):
        """Pipeline de preprocesamiento completo"""
        df = self.obtener_datos_filtrados(instrumento, origen, **kwargs)
        if df.empty:
            return None

        # Ordenar por fecha
        columna_fecha = self.detectar_columnas_temporales(df)[0] if self.detectar_columnas_temporales(df) else 'FECHA'
        try:
            df[columna_fecha] = pd.to_datetime(df[columna_fecha], dayfirst=True, errors='coerce')
            df = df.sort_values(columna_fecha)
        except:
            print(f"Error al procesar la columna de fecha: {columna_fecha}")

        # Imputación
        df = self.imputar_valores_faltantes(df, metodo=metodo_imputacion)

        # Tratamiento de outliers
        df = self.tratar_outliers(df, metodo=metodo_outliers)

        # Ingeniería de características
        df = self.ingenieria_caracteristicas(df, ventana=ventana_caracteristicas)

        # Normalización
        df = self.normalizar_datos(df, metodo=metodo_normalizacion)

        return df

    def visualizar_preprocesamiento(self, df, variable=None, columna_fecha=None):
        """Visualiza datos preprocesados con Plotly"""
        if df.empty:
            return None

        if columna_fecha is None:
            columnas_temporales = self.detectar_columnas_temporales(df)
            if not columnas_temporales:
                print("No se encontraron columnas temporales")
                return None
            columna_fecha = columnas_temporales[0]

        numericas = self.detectar_columnas_numericas(df)
        if variable and variable != "Todos" and variable in numericas:
            numericas = [variable]

        if not numericas:
            print("No se encontraron columnas numéricas")
            return None

        fig = make_subplots(rows=len(numericas), cols=2,
                           subplot_titles=[f"Serie Temporal - {col}" for col in numericas] +
                                         [f"Distribución - {col}" for col in numericas],
                           row_heights=[0.5]*len(numericas))

        for i, col in enumerate(numericas):
            # Serie temporal
            fig.add_trace(
                go.Scatter(x=df[columna_fecha], y=df[col], mode='lines', name=col),
                row=i+1, col=1
            )
            # Histograma
            fig.add_trace(
                go.Histogram(x=df[col], nbinsx=30, name=f"{col} (Hist)"),
                row=i+1, col=2
            )

        fig.update_layout(height=300*len(numericas), width=1000,
                         title_text="Visualización de Datos Preprocesados",
                         showlegend=True)
        fig.update_xaxes(title_text="Fecha", row=1, col=1)
        fig.update_yaxes(title_text="Valor", row=1, col=1)
        fig.update_xaxes(title_text="Valor", row=1, col=2)
        fig.update_yaxes(title_text="Frecuencia", row=1, col=2)
        fig.show()

        return fig

# =============================================================================
# INTERFAZ GRÁFICA PARA PREPROCESAMIENTO
# =============================================================================

# Crear instancia del preprocesador
try:
    preprocesador = PreprocesamientoDatos(datos_csv, datos_xlsx)
except NameError:
    display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ Error: Los datos 'datos_csv' y 'datos_xlsx' no están definidos.</div>"))
    preprocesador = None

# Selectores comunes
instrumento_dropdown = widgets.Dropdown(
    options=["Puntos Fijos", "Piezómetros Eléctricos", "Piezómetros Casagrande", "Inclinómetros",
             "Celdas de Asentamiento", "Freatímetros", "Extensómetros"],
    value="Puntos Fijos",
    description="Instrumento:"
)
origen_dropdown = widgets.Dropdown(
    options=["CSV", "XLSX"],
    value="CSV",
    description="Origen:"
)

# Selectores por instrumento
margen_dropdown = widgets.Dropdown(description="Margen:", layout={'width': '200px'})
punto_dropdown = widgets.Dropdown(description="Punto Fijo:", layout={'width': '200px'})
variable_pf_dropdown = widgets.Dropdown(description="Variable:", layout={'width': '200px'})
anio_pf_dropdown = widgets.Dropdown(description="Año:", layout={'width': '150px'})

progresiva_dropdown = widgets.Dropdown(description="Progresiva:", layout={'width': '200px'})
piezometro_dropdown = widgets.Dropdown(description="Piezómetro:", layout={'width': '200px'})
variable_pe_dropdown = widgets.Dropdown(description="Variable:", layout={'width': '200px'})
anio_pe_dropdown = widgets.Dropdown(description="Año:", layout={'width': '150px'})

margen_cg_dropdown = widgets.Dropdown(description="Margen:", layout={'width': '200px'})
pz_cg_dropdown = widgets.Dropdown(description="Piezómetro:", layout={'width': '200px'})
variable_cg_dropdown = widgets.Dropdown(description="Variable:", layout={'width': '200px'})
anio_cg_dropdown = widgets.Dropdown(description="Año:", layout={'width': '150px'})

inclinometro_dropdown = widgets.Dropdown(description="Inclinómetro:", layout={'width': '200px'})
anio_inc_dropdown = widgets.Dropdown(description="Año:", layout={'width': '150px'})
eje_dropdown = widgets.Dropdown(
    options=["Todos", "A+", "A-", "B+", "B-"],
    value="Todos",
    description="Eje:",
    layout={'width': '150px'}
)

progresiva_ca_dropdown = widgets.Dropdown(description="Progresiva:", layout={'width': '200px'})
celda_dropdown = widgets.Dropdown(description="Celda:", layout={'width': '200px'})
variable_ca_dropdown = widgets.Dropdown(description="Variable:", layout={'width': '200px'})
anio_ca_dropdown = widgets.Dropdown(description="Año:", layout={'width': '150px'})

freatimetro_dropdown = widgets.Dropdown(description="Freatímetro:", layout={'width': '200px'})
variable_fr_dropdown = widgets.Dropdown(description="Variable:", layout={'width': '200px'})
anio_fr_dropdown = widgets.Dropdown(description="Año:", layout={'width': '150px'})

extensometro_dropdown = widgets.Dropdown(description="Extensómetro:", layout={'width': '200px'})
variable_ex_dropdown = widgets.Dropdown(description="Variable:", layout={'width': '200px'})
anio_ex_dropdown = widgets.Dropdown(description="Año:", layout={'width': '150px'})

# Selectores de preprocesamiento
metodo_imputacion_dropdown = widgets.Dropdown(
    options=['linear', 'mean', 'median'],
    value='linear',
    description='Imputación:',
    layout={'width': '200px'}
)
metodo_outliers_dropdown = widgets.Dropdown(
    options=['clip', 'remove'],
    value='clip',
    description='Outliers:',
    layout={'width': '200px'}
)
metodo_normalizacion_dropdown = widgets.Dropdown(
    options=['standard', 'minmax', 'none'],
    value='standard',
    description='Normalización:',
    layout={'width': '200px'}
)
ventana_caracteristicas_slider = widgets.IntSlider(
    value=7, min=3, max=30, step=1,
    description='Ventana:',
    layout={'width': '300px'}
)

# Botones y salida
boton_preprocesar = widgets.Button(
    description='⚙️ Preprocesar',
    button_style='primary',
    icon='cogs',
    layout=widgets.Layout(width='120px')
)
output_preprocesamiento = widgets.Output()

# Funciones para actualizar opciones
def actualizar_opciones_pf(change=None):
    if not preprocesador:
        return
    origen = origen_dropdown.value
    try:
        df_mi = (datos_csv["puntos_fijos_mi"] if origen == "CSV" else datos_xlsx["puntos_fijos_mi"]).copy()
        df_md = (datos_csv["puntos_fijos_md"] if origen == "CSV" else datos_xlsx["puntos_fijos_md"]).copy()
        datasets = {"Margen Izquierda (MI)": df_mi, "Margen Derecha (MD)": df_md}
        datasets = {k: v for k, v in datasets.items() if not v.empty}
        if not datasets:
            margen_dropdown.options = []
            punto_dropdown.options = []
            variable_pf_dropdown.options = []
            anio_pf_dropdown.options = []
            return
        margen_dropdown.options = ["Todos"] + list(datasets.keys())
        margen = margen_dropdown.value or "Todos"
        if margen == "Todos":
            df = pd.concat([df for df in datasets.values() if not df.empty], ignore_index=True)
        else:
            df = datasets[margen].copy()
        df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
        columnas_excluir = ['FECHA', 'INSTRUMENTO', 'MARGEN']
        variable_pf_dropdown.options = ["Todos"] + [col for col in df.select_dtypes(include='number').columns if col not in columnas_excluir]
        punto_dropdown.options = ["Todos"] + sorted(df['INSTRUMENTO'].dropna().unique())
        anio_pf_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]
    except Exception as e:
        print(f"Error al actualizar opciones de Puntos Fijos: {str(e)}")

def actualizar_opciones_pe(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["piezometros_electricos"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_electricos"]).copy()
        if df.empty or 'PROGRESIVA' not in df.columns:
            progresiva_dropdown.options = []
            piezometro_dropdown.options = []
            variable_pe_dropdown.options = []
            anio_pe_dropdown.options = []
            return
        df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
        progresiva_dropdown.options = ["Todos"] + sorted(df['PROGRESIVA'].dropna().unique())
        if progresiva_dropdown.options:
            progresiva_dropdown.value = progresiva_dropdown.options[0]
        actualizar_piezometros_pe()
        columnas_excluir = ['FECHA', 'PROGRESIVA', 'PIEZOMETRO']
        variable_pe_dropdown.options = ["Todos"] + [c for c in df.columns if c not in columnas_excluir]
        if variable_pe_dropdown.options:
            variable_pe_dropdown.value = variable_pe_dropdown.options[0]
        anio_pe_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]
    except Exception as e:
        print(f"Error al actualizar opciones de Piezómetros Eléctricos: {str(e)}")

def actualizar_piezometros_pe(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["piezometros_electricos"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_electricos"]).copy()
        if df.empty or 'PROGRESIVA' not in df.columns or 'PIEZOMETRO' not in df.columns:
            piezometro_dropdown.options = []
            return
        if progresiva_dropdown.value == "Todos":
            piezos = sorted(df['PIEZOMETRO'].dropna().unique())
        else:
            piezos = sorted(df[df['PROGRESIVA'] == progresiva_dropdown.value]['PIEZOMETRO'].dropna().unique())
        piezometro_dropdown.options = ["Todos"] + list(piezos)
        if piezometro_dropdown.options:
            piezometro_dropdown.value = piezometro_dropdown.options[0]
    except Exception as e:
        print(f"Error al actualizar piezómetros: {str(e)}")

def actualizar_opciones_cg(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["piezometros_casagrande"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_casagrande"]).copy()
        if df.empty or 'MARGEN' not in df.columns:
            margen_cg_dropdown.options = []
            pz_cg_dropdown.options = []
            variable_cg_dropdown.options = []
            anio_cg_dropdown.options = []
            return
        df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
        margen_cg_dropdown.options = ["Todos"] + sorted(df['MARGEN'].dropna().unique())
        if margen_cg_dropdown.options:
            margen_cg_dropdown.value = margen_cg_dropdown.options[0]
        actualizar_piezometros_cg()
        columnas_excluir = ['FECHA', 'MARGEN', 'PIEZOMETRO']
        variable_cg_dropdown.options = ["Todos"] + [c for c in df.columns if c not in columnas_excluir]
        if variable_cg_dropdown.options:
            variable_cg_dropdown.value = variable_cg_dropdown.options[0]
        anio_cg_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]
    except Exception as e:
        print(f"Error al actualizar opciones de Piezómetros Casagrande: {str(e)}")

def actualizar_piezometros_cg(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["piezometros_casagrande"] if origen_dropdown.value == "CSV" else datos_xlsx["piezometros_casagrande"]).copy()
        if df.empty or 'MARGEN' not in df.columns or 'PIEZOMETRO' not in df.columns:
            pz_cg_dropdown.options = []
            return
        if margen_cg_dropdown.value == "Todos":
            piezos = sorted(df['PIEZOMETRO'].dropna().unique())
        else:
            piezos = sorted(df[df['MARGEN'] == margen_cg_dropdown.value]['PIEZOMETRO'].dropna().unique())
        pz_cg_dropdown.options = ["Todos"] + list(piezos)
        if pz_cg_dropdown.options:
            pz_cg_dropdown.value = pz_cg_dropdown.options[0]
    except Exception as e:
        print(f"Error al actualizar piezómetros Casagrande: {str(e)}")

def actualizar_opciones_inc(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["inclinometros"] if origen_dropdown.value == "CSV" else datos_xlsx["inclinometros"]).copy()
        if df.empty:
            inclinometro_dropdown.options = []
            anio_inc_dropdown.options = []
            return
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        inclinometro_dropdown.options = ["Todos"] + sorted(df['Inclinometro'].dropna().unique())
        anio_inc_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['Fecha'].dt.year.dropna().unique())]
    except Exception as e:
        print(f"Error al actualizar opciones de Inclinómetros: {str(e)}")

def actualizar_opciones_ca(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["asentamiento"] if origen_dropdown.value == "CSV" else datos_xlsx["asentamiento"]).copy()
        if df.empty:
            progresiva_ca_dropdown.options = []
            celda_dropdown.options = []
            variable_ca_dropdown.options = []
            anio_ca_dropdown.options = []
            return
        df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
        progresiva_ca_dropdown.options = ["Todos"] + sorted(df['PROGRESIVA'].dropna().unique())
        actualizar_celdas_ca()
        columnas_excluir = ['FECHA', 'PROGRESIVA', 'CELDA_DE_ASENTAMIENTO']
        variable_ca_dropdown.options = ["Todos"] + [c for c in df.select_dtypes(include='number').columns if c not in columnas_excluir]
        anio_ca_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]
    except Exception as e:
        print(f"Error al actualizar opciones de Celdas de Asentamiento: {str(e)}")

def actualizar_celdas_ca(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["asentamiento"] if origen_dropdown.value == "CSV" else datos_xlsx["asentamiento"]).copy()
        if df.empty or 'PROGRESIVA' not in df.columns or 'CELDA_DE_ASENTAMIENTO' not in df.columns:
            celda_dropdown.options = []
            return
        if progresiva_ca_dropdown.value == "Todos":
            celdas = sorted(df['CELDA_DE_ASENTAMIENTO'].dropna().unique())
        else:
            celdas = sorted(df[df['PROGRESIVA'] == progresiva_ca_dropdown.value]['CELDA_DE_ASENTAMIENTO'].dropna().unique())
        celda_dropdown.options = ["Todos"] + list(celdas)
        if celda_dropdown.options:
            celda_dropdown.value = celda_dropdown.options[0]
    except Exception as e:
        print(f"Error al actualizar celdas: {str(e)}")

def actualizar_opciones_fr(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["freatimetros"] if origen_dropdown.value == "CSV" else datos_xlsx["freatimetros"]).copy()
        if df.empty:
            freatimetro_dropdown.options = []
            variable_fr_dropdown.options = []
            anio_fr_dropdown.options = []
            return
        df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
        freatimetro_dropdown.options = ["Todos"] + sorted(df['FREATIMETRO'].dropna().unique())
        columnas_excluir = ['FECHA', 'FREATIMETRO']
        variable_fr_dropdown.options = ["Todos"] + [c for c in df.select_dtypes(include='number').columns if c not in columnas_excluir]
        anio_fr_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]
    except Exception as e:
        print(f"Error al actualizar opciones de Freatímetros: {str(e)}")

def actualizar_opciones_ex(change=None):
    if not preprocesador:
        return
    try:
        df = (datos_csv["extensometro"] if origen_dropdown.value == "CSV" else datos_xlsx["extensometro"]).copy()
        if df.empty:
            extensometro_dropdown.options = []
            variable_ex_dropdown.options = []
            anio_ex_dropdown.options = []
            return
        df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
        extensometro_dropdown.options = ["Todos"] + sorted(df['EXTENSOMETRO'].dropna().unique())
        columnas_excluir = ['FECHA', 'EXTENSOMETRO']
        variable_ex_dropdown.options = ["Todos"] + [c for c in df.select_dtypes(include='number').columns if c not in columnas_excluir]
        anio_ex_dropdown.options = ["Todos"] + [str(y) for y in sorted(df['FECHA'].dt.year.dropna().unique())]
    except Exception as e:
        print(f"Error al actualizar opciones de Extensómetros: {str(e)}")

def actualizar_controles_visibles(change=None):
    if not preprocesador:
        return
    tipo = instrumento_dropdown.value
    for w in [margen_dropdown, punto_dropdown, variable_pf_dropdown, anio_pf_dropdown,
              progresiva_dropdown, piezometro_dropdown, variable_pe_dropdown, anio_pe_dropdown,
              margen_cg_dropdown, pz_cg_dropdown, variable_cg_dropdown, anio_cg_dropdown,
              inclinometro_dropdown, anio_inc_dropdown, eje_dropdown,
              progresiva_ca_dropdown, celda_dropdown, variable_ca_dropdown, anio_ca_dropdown,
              freatimetro_dropdown, variable_fr_dropdown, anio_fr_dropdown,
              extensometro_dropdown, variable_ex_dropdown, anio_ex_dropdown]:
        w.layout.display = 'none'

    if tipo == "Puntos Fijos":
        margen_dropdown.layout.display = 'flex'
        punto_dropdown.layout.display = 'flex'
        variable_pf_dropdown.layout.display = 'flex'
        anio_pf_dropdown.layout.display = 'flex'
        actualizar_opciones_pf()
    elif tipo == "Piezómetros Eléctricos":
        progresiva_dropdown.layout.display = 'flex'
        piezometro_dropdown.layout.display = 'flex'
        variable_pe_dropdown.layout.display = 'flex'
        anio_pe_dropdown.layout.display = 'flex'
        actualizar_opciones_pe()
    elif tipo == "Piezómetros Casagrande":
        margen_cg_dropdown.layout.display = 'flex'
        pz_cg_dropdown.layout.display = 'flex'
        variable_cg_dropdown.layout.display = 'flex'
        anio_cg_dropdown.layout.display = 'flex'
        actualizar_opciones_cg()
    elif tipo == "Inclinómetros":
        inclinometro_dropdown.layout.display = 'flex'
        anio_inc_dropdown.layout.display = 'flex'
        eje_dropdown.layout.display = 'flex'
        actualizar_opciones_inc()
    elif tipo == "Celdas de Asentamiento":
        progresiva_ca_dropdown.layout.display = 'flex'
        celda_dropdown.layout.display = 'flex'
        variable_ca_dropdown.layout.display = 'flex'
        anio_ca_dropdown.layout.display = 'flex'
        actualizar_opciones_ca()
    elif tipo == "Freatímetros":
        freatimetro_dropdown.layout.display = 'flex'
        variable_fr_dropdown.layout.display = 'flex'
        anio_fr_dropdown.layout.display = 'flex'
        actualizar_opciones_fr()
    elif tipo == "Extensómetros":
        extensometro_dropdown.layout.display = 'flex'
        variable_ex_dropdown.layout.display = 'flex'
        anio_ex_dropdown.layout.display = 'flex'
        actualizar_opciones_ex()

def ejecutar_preprocesamiento(b):
    if not preprocesador:
        display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ Error: Preprocesador no inicializado.</div>"))
        return
    with output_preprocesamiento:
        clear_output(wait=True)
        instrumento = instrumento_dropdown.value
        origen = origen_dropdown.value
        metodo_imputacion = metodo_imputacion_dropdown.value
        metodo_outliers = metodo_outliers_dropdown.value
        metodo_normalizacion = metodo_normalizacion_dropdown.value
        ventana_caracteristicas = ventana_caracteristicas_slider.value

        kwargs = {}
        variable = None
        if instrumento == "Puntos Fijos":
            kwargs = {'margen': margen_dropdown.value, 'punto': punto_dropdown.value, 'anio': anio_pf_dropdown.value}
            variable = variable_pf_dropdown.value
        elif instrumento == "Piezómetros Eléctricos":
            kwargs = {'progresiva': progresiva_dropdown.value, 'piezometro': piezometro_dropdown.value, 'anio': anio_pe_dropdown.value}
            variable = variable_pe_dropdown.value
        elif instrumento == "Piezómetros Casagrande":
            kwargs = {'margen': margen_cg_dropdown.value, 'piezometro': pz_cg_dropdown.value, 'anio': anio_cg_dropdown.value}
            variable = variable_cg_dropdown.value
        elif instrumento == "Inclinómetros":
            kwargs = {'inclinometro': inclinometro_dropdown.value, 'anio': anio_inc_dropdown.value, 'eje': eje_dropdown.value}
        elif instrumento == "Celdas de Asentamiento":
            kwargs = {'progresiva': progresiva_ca_dropdown.value, 'celda': celda_dropdown.value, 'anio': anio_ca_dropdown.value}
            variable = variable_ca_dropdown.value
        elif instrumento == "Freatímetros":
            kwargs = {'freatimetro': freatimetro_dropdown.value, 'anio': anio_fr_dropdown.value}
            variable = variable_fr_dropdown.value
        elif instrumento == "Extensómetros":
            kwargs = {'extensometro': extensometro_dropdown.value, 'anio': anio_ex_dropdown.value}
            variable = variable_ex_dropdown.value

        try:
            df = preprocesador.preprocesar_datos(
                instrumento, origen,
                metodo_imputacion=metodo_imputacion,
                metodo_outliers=metodo_outliers,
                metodo_normalizacion=metodo_normalizacion,
                ventana_caracteristicas=ventana_caracteristicas,
                **kwargs
            )
            if df is None or df.empty:
                display(HTML("<div style='color:#b71c1c;font-weight:bold;'>⚠️ No hay datos disponibles para el preprocesamiento.</div>"))
                return

            # Mostrar resumen
            print(f"✅ Datos preprocesados para {instrumento} ({origen})")
            print(f"Registros: {len(df)}")
            print(f"Columnas: {', '.join(df.columns)}")
            display(df.head().round(3))

            # Visualizar datos preprocesados
            preprocesador.visualizar_preprocesamiento(df, variable=variable)

        except Exception as e:
            display(HTML(f"<div style='color:#b71c1c;'>❌ Error en el preprocesamiento: {str(e)}</div>"))

# Conectar eventos
instrumento_dropdown.observe(actualizar_controles_visibles, names='value')
origen_dropdown.observe(actualizar_controles_visibles, names='value')
margen_dropdown.observe(actualizar_opciones_pf, names='value')
progresiva_dropdown.observe(actualizar_piezometros_pe, names='value')
margen_cg_dropdown.observe(actualizar_piezometros_cg, names='value')
progresiva_ca_dropdown.observe(actualizar_celdas_ca, names='value')
boton_preprocesar.on_click(ejecutar_preprocesamiento)

# Mostrar interfaz
display(HTML("<hr style='margin:20px 0 15px 0;'>"))
display(HTML("""
<div style='margin-bottom:15px;'>
    <h2 style='color:#1976d2;margin:0 0 4px 0;'>⚙️ Preprocesamiento de Datos</h2>
    <span style='color:#555;'>Limpieza, normalización e ingeniería de características para datos de instrumentos</span>
</div>
"""))
display(HTML("<b>1. Selecciona el instrumento y origen:</b>"))
display(widgets.HBox([instrumento_dropdown, origen_dropdown]))
display(HTML("<b>2. Selecciona los parámetros específicos:</b>"))
display(widgets.VBox([
    widgets.HBox([margen_dropdown, punto_dropdown, variable_pf_dropdown, anio_pf_dropdown]),
    widgets.HBox([progresiva_dropdown, piezometro_dropdown, variable_pe_dropdown, anio_pe_dropdown]),
    widgets.HBox([margen_cg_dropdown, pz_cg_dropdown, variable_cg_dropdown, anio_cg_dropdown]),
    widgets.HBox([inclinometro_dropdown, anio_inc_dropdown, eje_dropdown]),
    widgets.HBox([progresiva_ca_dropdown, celda_dropdown, variable_ca_dropdown, anio_ca_dropdown]),
    widgets.HBox([freatimetro_dropdown, variable_fr_dropdown, anio_fr_dropdown]),
    widgets.HBox([extensometro_dropdown, variable_ex_dropdown, anio_ex_dropdown])
]))
display(HTML("<b>3. Configura las opciones de preprocesamiento:</b>"))
display(widgets.HBox([
    metodo_imputacion_dropdown,
    metodo_outliers_dropdown,
    metodo_normalizacion_dropdown,
    ventana_caracteristicas_slider
]))
display(widgets.HBox([boton_preprocesar]))
display(output_preprocesamiento)

# Inicializar
if preprocesador:
    actualizar_controles_visibles()

Output()